# 05 · The router (Version 4): the cheapest model that can do the job

```
message → ML model ──very sure + simple──────────→ ready-made answer   (free, ~1 ms)
              │
              └─fairly sure + simple──→ local LLM → short answer         (free, ~1 s)
              │
              └─action / angry / unsure──→ cloud LLM agent + tools      (paid, a few s)
```

**My plan:**
1. See which layer each message would go to (free, no LLM)
2. Try the router on single messages
3. Compare V2 (big LLM for everything) vs V4 (router) on a mixed inbox

In [ ]:
import sys
sys.path.append("..")   # so Python can find the budget_agent folder

import pandas as pd
pd.set_option("display.max_colwidth", 80)
from dotenv import load_dotenv
load_dotenv("../.env")

from budget_agent import config, data
from budget_agent.router import Router

router = Router()
print("Local LLM ready:", router.local_ready())

## Step 1: Who would answer? (no LLM is called here)

In [ ]:
for text in ["what payment methods do u accept",
             "where is my order NM10005",
             "kitne din me delivery hoti hai",
             "refund NM10033 the earbuds broke",
             "this is the 3rd time my delivery is late, I want a manager"]:
    layer, intent, confidence = router.choose_layer(text)
    print(f"{layer:10} {intent:24} {confidence:4.0%}   {text}")

In [ ]:
test = data.load_messages().query("split == 'test'")
layers = [router.choose_layer(text)[0] for text in test.text]
pd.Series(layers).value_counts(normalize=True)

On normal messages, a big share never needs the cloud LLM at all.
(Refunds, cancels, complaints etc. always go to the cloud LLM, on purpose, because they change real things.)

## Step 2: Try the router for real

In [ ]:
def show(r):
    print(f"[{r.layer}] intent={r.intent} ({r.confidence:.0%})  {r.seconds:.3f}s  ${r.cost_usd:.4f}")
    print("  ", r.reply, "\n")

show(router.handle("what payment methods do u accept"))
show(router.handle("where is my order NM10005"))

In [ ]:
show(router.handle("my order NM10033 arrived broken, refund please"))

## Step 3: Version 2 vs Version 4 on a mixed inbox

This calls the cloud LLM many times, so I start small (`--n 30` costs well under $1).
I run it from the terminal so it saves a CSV the app can show:

```
python run.py benchmark --n 30
```

In [ ]:
results = pd.read_csv(config.DATA_DIR / "benchmark_results.csv")
summary = pd.DataFrame({
    "total cost ($)": [results.v2_cost.sum(), results.v4_cost.sum()],
    "average seconds": [results.v2_seconds.mean(), results.v4_seconds.mean()],
}, index=["V2: big LLM for everything", "V4: router"])
summary.round(3)

In [ ]:
print(f"V4 is {results.v2_cost.sum() / results.v4_cost.sum():.1f}x cheaper")
results.v4_layer.value_counts()

In [ ]:
# Read some replies side by side, to check quality didn't drop
results[["message", "v4_layer", "v2_reply", "v4_reply"]].sample(5, random_state=1)

## What I learned
- Most of the work in a support inbox is simple. A tiny ML model handles it in milliseconds for free.
- The router only pays for the cloud LLM when it's really needed: actions, angry customers, unclear messages.
- The confidence threshold is a **dial**: higher = safer but more expensive, lower = cheaper but riskier.
- This is **inference engineering**: choosing the right model for each request, not one giant model for everything.